# Aula 10: pipeline de dados com PySpark

Atividade prática de extração, transformação e unificação de dados, usando
o Spark com a base de fraude em cartão (a mesma das Aulas 2, 5, 6, 7, 8 e 9),
agora em escala de 1 milhão de transações.

O notebook está dividido em passos. Rode as células em ordem, uma por vez.
Onde aparecer `# TODO`, complete a linha antes de rodar. Se travar, escreva
no chat com o número do passo, não precisa esperar até o final.

**Antes de começar:** os arquivos CSV desta atividade estão na pasta
`docs/aula-08/dados/csv`. Ajuste `PASTA_DADOS` na célula abaixo se o
notebook estiver em outro lugar do seu Drive.

## Passo 0 · Preparar o ambiente

O Google Colab não vem com Spark instalado. As duas células abaixo
resolvem isso, e rodam só uma vez por sessão (se o Colab desconectar,
rode de novo desde o início).

In [1]:
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install pyspark -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("aula10").getOrCreate()
spark

In [2]:
import gdown
import os

# Dicionário com IDs dos arquivos no Google Drive
files = {
    'cartoes.csv': '13Xf9ILhqxaIcvjq0vENJFCPD-HX-FLjM',
    'categorias.csv': '1NeAB_U66QnzRHZ1tJesP5lZ0jccXLAke',
    'clientes.csv': '1zgr0k17FdKp5aRkSd96qtLyJ9x0b4bFv',
    'contestacoes.csv': '1lSpAe8stD3JEMDJMFbiiYrQH0jYtHzT8',
    'lojistas.csv': '1vydqkbd2fONvNAfNKon9izr0pqaGBqhy',
    'transacoes.csv': '1OR3lgLjqVQZoajau2wrSExWPYc8xUsFS'
}

# Criar pasta para os CSVs para manter a estrutura
os.makedirs('csv', exist_ok=True)

print("Baixando arquivos...")
for filename, file_id in files.items():
    url = f'https://drive.google.com/uc?id={file_id}'
    output = f'csv/{filename}' if filename.endswith('.csv') else filename
    gdown.download(url, output, quiet=True)
    print(f"- {filename} baixado.")

PASTA_DADOS = "/content/csv"

Baixando arquivos...
- cartoes.csv baixado.
- categorias.csv baixado.
- clientes.csv baixado.
- contestacoes.csv baixado.
- lojistas.csv baixado.
- transacoes.csv baixado.


## Passo 1 · Extract

As seis tabelas do caso de fraude, direto do CSV. É a mesma leitura da
Aula 8, só que com `spark.read.csv` no lugar de `pd.read_csv`.

In [12]:
from pyspark.sql import functions as F

dict_dfs = {}

categorias   = spark.read.csv(f"{PASTA_DADOS}/categorias.csv",   header=True, inferSchema=True)
dict_dfs['categorias'] = categorias

clientes     = spark.read.csv(f"{PASTA_DADOS}/clientes.csv",     header=True, inferSchema=True)
dict_dfs['clientes'] = clientes

lojistas     = spark.read.csv(f"{PASTA_DADOS}/lojistas.csv",     header=True, inferSchema=True)
dict_dfs['lojistas'] = lojistas

cartoes      = spark.read.csv(f"{PASTA_DADOS}/cartoes.csv",      header=True, inferSchema=True)
dict_dfs['cartoes'] = cartoes

transacoes   = spark.read.csv(f"{PASTA_DADOS}/transacoes.csv",   header=True, inferSchema=True)
dict_dfs['transacoes'] = transacoes

contestacoes = spark.read.csv(f"{PASTA_DADOS}/contestacoes.csv", header=True, inferSchema=True)
dict_dfs['contestacoes'] = contestacoes

for name, df in dict_dfs.items():
    print(f"Schema do DataFrame {name}")
    df.printSchema()
    print()


Schema do DataFrame categorias
root
 |-- id_categoria: integer (nullable = true)
 |-- nome: string (nullable = true)


Schema do DataFrame clientes
root
 |-- id_cliente: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- cpf: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_nascimento: date (nullable = true)
 |-- data_cadastro: date (nullable = true)
 |-- telefone: string (nullable = true)
 |-- score_risco: double (nullable = true)


Schema do DataFrame lojistas
root
 |-- id_lojista: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- id_categoria: integer (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_cadastro: date (nullable = true)


Schema do DataFrame cartoes
root
 |-- id_cartao: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- bandeira: string (nullable = true)
 |-- final_cartao: integer (nullable = true)
 |

## Passo 2 · Transform

Antes de unir as tabelas, limpe `transacoes` com os três critérios de
qualidade já vistos na Aula 9: sem valor nulo nos campos obrigatórios, sem
`id_transacao` duplicado, e sem valor menor ou igual a zero.

In [13]:
transacoes_limpo = (
    transacoes
    .dropna(subset=["id_transacao", "id_cartao", "id_lojista", "valor"])
    .dropDuplicates(["id_transacao"])
    .filter(F.col("valor") > 0)
)

print("Linhas antes da limpeza:", transacoes.count())
print("Linhas depois da limpeza:", transacoes_limpo.count())

Linhas antes da limpeza: 1000090
Linhas depois da limpeza: 1000090


## Passo 3 · Unify, primeira tentativa

Junte `transacoes_limpo` com as outras quatro tabelas, pela chave que liga
cada par. Depois, rode a célula seguinte para conferir o nome do cliente
numa das transações.

In [14]:
completo = (
    transacoes_limpo
    .join(cartoes, on="id_cartao", how="inner")
    .join(clientes, on="id_cliente", how="inner")
    .join(lojistas, on="id_lojista", how="inner")
    .join(categorias, on="id_categoria", how="left")
)

In [8]:
completo.select("nome").show(5)

{"ts": "2026-09-10 11:31:02.678", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[AMBIGUOUS_REFERENCE] Reference `nome` is ambiguous, could be: [`nome`, `nome`, `nome`]. SQLSTATE: 42704", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "AMBIGUOUS_REFERENCE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o92.select.\n: org.apache.spark.sql.AnalysisException: [AMBIGUOUS_REFERENCE] Reference `nome` is ambiguous, could be: [`nome`, `nome`, `nome`]. SQLSTATE: 42704\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:2163)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:363)\n\tat org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:164)\n\tat org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper

AnalysisException: [AMBIGUOUS_REFERENCE] Reference `nome` is ambiguous, could be: [`nome`, `nome`, `nome`]. SQLSTATE: 42704

Se a célula acima deu um erro mencionando `nome` como referência ambígua
(*ambiguous reference*), não é bug seu. `clientes`, `lojistas` e
`categorias` têm, cada uma, uma coluna chamada `nome`, com um significado
diferente em cada tabela. O Spark não sabe qual das três você quis dizer.

Isso é exatamente o problema de **consistência** que a Aula 9 apresentou:
o mesmo nome de campo significando coisas diferentes em tabelas diferentes.

A saída é renomear a coluna `nome` de cada tabela antes de unir, com
`.alias(...)`, para que o resultado final tenha `nome_cliente`,
`nome_lojista` e `nome_categoria`, cada um com um nome só seu.

In [15]:
clientes_r = clientes.select(
    "id_cliente",
    F.col("nome").alias("nome_cliente"),
)

lojistas_r = lojistas.select(
    "id_lojista",
    "id_categoria",
    F.col("nome").alias("nome_lojista"),
)

categorias_r = categorias.select(
    "id_categoria",
    F.col("nome").alias("nome_categoria"),
)

## Passo 3 (continuação) · Unify, com os nomes corrigidos

Refaça a união, agora usando `clientes_r`, `lojistas_r` e `categorias_r`
no lugar das tabelas originais. De `cartoes`, use só as três colunas
necessárias, para não repetir nenhuma outra coluna por acidente.

In [16]:
base_unificada = (
    transacoes_limpo
    .join(cartoes.select("id_cartao", "id_cliente", "bandeira"), on="id_cartao", how="inner")
    .join(clientes_r, on="id_cliente", how="inner")
    .join(lojistas_r, on="id_lojista", how="inner")
    .join(categorias_r, on="id_categoria", how="left")
)

base_unificada.select("id_transacao", "valor", "nome_cliente", "nome_lojista", "nome_categoria").show(5)

+------------+------+--------------------+---------------+--------------+
|id_transacao| valor|        nome_cliente|   nome_lojista|nome_categoria|
+------------+------+--------------------+---------------+--------------+
|         148| 252.4|      Cauê Gonçalves| Praça Livraria|      Livraria|
|         463| 37.23|     Hellena Machado|Horizonte Saúde|      Farmácia|
|         471| 48.62|     Cecília Rezende|  Central Lazer|         Lazer|
|         496|210.74|       Luísa Machado|  Aurora Cursos|      Educação|
|         833|143.44|Dr. José Miguel N...| Serra Serviços|      Serviços|
+------------+------+--------------------+---------------+--------------+
only showing top 5 rows


## Passo 4 · Load

Grave o resultado final e responda as duas perguntas de conferência.

In [17]:
base_unificada.write.mode("overwrite").parquet("/content/base_unificada")

# Pergunta 1: quantas linhas sobraram depois da limpeza?
print("Linhas depois da limpeza:", base_unificada.count())

# Pergunta 2: qual lojista teve o maior valor total transacionado?
(
    base_unificada
    .groupBy("nome_lojista")
    .sum("valor")
    .orderBy(F.desc("sum(valor)"))
    .show(5)
)

Linhas depois da limpeza: 1000090
+-----------------+------------------+
|     nome_lojista|        sum(valor)|
+-----------------+------------------+
|   Horizonte Moda|1262206.3300000003|
|  Cidade Livraria|1255067.0199999986|
|Aurora Auto Posto|        1220325.81|
|   Horizonte Tech|1211078.1600000008|
|   Rio Mobilidade|1194597.9800000037|
+-----------------+------------------+
only showing top 5 rows


## Pronto

Confira se as quatro etapas rodaram sem erro (extract, transform, unify e
load), e anote as respostas das duas perguntas de conferência. Seu
instrutor vai avisar como entregar. Se alguma célula travou, escreva no
chat com o número do passo.